# 02 — XGBoost & LightGBM RUL baselines

From the gold features built in notebook 01, we train two gradient-boosting baselines for Remaining Useful Life regression on the FD001 sub-dataset and compare them on the **official NASA test set** (one row per engine at its last observed cycle, scored against the ground-truth `RUL_FD001.txt`).

**Methodological choices (and why):**

1. **Piecewise-linear RUL clipping at 125** (Heimes 2008). Engines spend most of their life in a healthy regime — letting the target run up to 350+ cycles makes the model waste capacity learning early-life noise.
2. **Engine-stratified validation**, *not row-stratified*. Mixing cycles of the same engine across train/val is the standard C-MAPSS leakage trap; we avoid it by holding out 20% of `unit_id`s.
3. **NASA scoring function** alongside RMSE — penalises late predictions ~30% more than early ones, matching how a maintenance engineer actually pays for being wrong.
4. **MLflow logging** for every run with model signature + input example, so the artifact is registry-ready.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from turboguard.data.cmapss import load_cmapss, add_rul_clipped
from turboguard.features.pipeline import FeatureConfig, build_features, load_gold, save_gold
from turboguard.models.rul.splits import engine_stratified_split, prepare_test_set
from turboguard.models.rul.baselines import train_xgboost, train_lightgbm
from turboguard.models.rul.nasa_score import nasa_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
GOLD_PATH = ROOT / 'data' / 'processed' / 'gold_FD001.parquet'

mlflow.set_tracking_uri(f'file:{(ROOT / "mlruns").as_posix()}')
mlflow.set_experiment('turboguard-FD001')
print(f'MLflow tracking: {mlflow.get_tracking_uri()}')

## 1. Load gold features (or build them)

If notebook 01 has already been run, the parquet exists. Otherwise we rebuild on the fly.

In [ ]:
fd001 = load_cmapss('FD001')

if GOLD_PATH.exists():
    gold = load_gold(GOLD_PATH)
    print(f'Loaded existing gold: {gold.shape[0]:,} rows × {gold.shape[1]} cols')
else:
    print('Building gold features (this takes ~15s)...')
    gold, _ = build_features(fd001.with_rul())
    save_gold(gold, GOLD_PATH)
    print(f'Saved {gold.shape[0]:,} rows × {gold.shape[1]} cols → {GOLD_PATH}')

gold = add_rul_clipped(gold, cap=125)
print(f'\nRUL distribution: max={gold.RUL.max()}, clipped (cap=125): {gold.RUL_clipped.max()}')

## 2. Build the official test set

One row per test engine at its **last observed cycle**, labeled with the ground-truth RUL from `RUL_FD001.txt`.

In [ ]:
X_test, y_test, _ = prepare_test_set(fd001)
print(f'Test set: {len(X_test)} engines | mean true RUL = {y_test.mean():.1f} | min/max RUL = {y_test.min()}/{y_test.max()}')

## 3. Engine-stratified split

In [ ]:
split = engine_stratified_split(gold, val_fraction=0.2, rng_seed=42)
print(f'Train: {len(split.X_train):,} rows from {len(set(split.groups_train))} engines')
print(f'Val  : {len(split.X_val):,} rows from {len(set(split.groups_val))} engines')
print(f'Features: {len(split.feature_cols)}')

## 4. Train XGBoost baseline

In [ ]:
xgb_result = train_xgboost(split, X_test=X_test, y_test=y_test, run_name='xgb-FD001-default')
print('XGBoost validation:', xgb_result.val_metrics)
print('XGBoost test      :', xgb_result.test_metrics)

## 5. Train LightGBM baseline

In [ ]:
lgb_result = train_lightgbm(split, X_test=X_test, y_test=y_test, run_name='lgbm-FD001-default')
print('LightGBM validation:', lgb_result.val_metrics)
print('LightGBM test      :', lgb_result.test_metrics)

## 6. Comparison vs. literature benchmark

Reference values for FD001 from the canonical surveys (Saxena et al. 2008; Babu et al. 2016; Zheng et al. 2017): RMSE ~13–18, NASA score 200–500 for classical ML.

In [ ]:
summary = pd.DataFrame(
    [
        {'model': 'naive (mean lifetime)', 'val_rmse': None, 'val_nasa': None,
         'test_rmse': None, 'test_nasa': 22000},  # ballpark from notebook 00
        {'model': 'XGBoost', **xgb_result.val_metrics, **xgb_result.test_metrics},
        {'model': 'LightGBM', **lgb_result.val_metrics, **lgb_result.test_metrics},
    ]
)
summary[['model', 'val_rmse', 'val_nasa_score', 'test_rmse', 'test_nasa_score']]

## 7. Predicted vs. true RUL on the test set

Picking the better-performing model and inspecting where it fails.

In [ ]:
best = lgb_result if lgb_result.test_metrics['test_nasa_score'] < xgb_result.test_metrics['test_nasa_score'] else xgb_result
best_name = 'LightGBM' if best is lgb_result else 'XGBoost'
y_pred = np.clip(best.model.predict(X_test[split.feature_cols]), 0, None)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(y_test, y_pred, alpha=0.7)
lims = [0, max(y_test.max(), y_pred.max()) + 5]
axes[0].plot(lims, lims, 'k--', alpha=0.5, label='ideal')
axes[0].set_xlabel('true RUL'); axes[0].set_ylabel('predicted RUL')
axes[0].set_title(f'{best_name} — predicted vs. true RUL on FD001 test')
axes[0].legend()

residuals = y_pred - y_test
axes[1].hist(residuals, bins=25, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('residual (pred − true)')
axes[1].set_ylabel('engines')
axes[1].set_title('Residuals — negative = early, positive = late (worse)')
fig.tight_layout()
print(f'Late predictions: {(residuals > 0).sum()}/{len(residuals)} engines')
print(f'Mean late residual : {residuals[residuals > 0].mean():.2f}')
print(f'Mean early residual: {residuals[residuals < 0].mean():.2f}')

## 8. Top-20 most important features

In [ ]:
if hasattr(best.model, 'feature_importances_'):
    imp = pd.Series(best.model.feature_importances_, index=split.feature_cols).sort_values(ascending=False)
else:
    imp = pd.Series(best.model.booster_.feature_importance(), index=split.feature_cols).sort_values(ascending=False)

top = imp.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values)
ax.set_title(f'{best_name} — top 20 features by importance')
fig.tight_layout()

---

## What's next: `03_lstm_rul.ipynb`

We'll re-target the same problem with a sliding-window LSTM in PyTorch, then compare against these tabular baselines on the same NASA scoring function. The LSTM should win on FD002/FD004 (where multi-condition operation makes sequence context matter); on FD001 it's typically a wash with gradient boosting.

## To inspect the runs

```bash
# from the project root, in the activated venv:
python -m mlflow ui --backend-store-uri ./mlruns
```

Then open http://localhost:5000.